# Sample Builder Outputs EDA

This notebook explores how SampleBuilder configurations (provided through datamodules) behave on the TACO v1.3 trace datasets (a selected set of shards, or all shards). We measure what kind of samples are planned for generation, and what kind of samples are actually generated at runtime (given trace data constraints).

A summary of findings for the whole dataset is presented at the end.

In [ ]:
import collections
import typing

import IPython.display as ipy_display
import matplotlib.pyplot as plt
import pandas as pd
import tqdm

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.organisms.datamodules.shortcuts as shortcuts
import pyine.organisms.datamodules.shortcuts_configs as shortcuts_configs
import pyine.organisms.datamodules.utils.samples as sample_utils
import pyine.utils.pydantic

plt.style.use("ggplot")

In [ ]:
# ------------ CHANGE THESE SETTINGS IF NEEDED ------------
source_dataset_name = "TACO"  # by default, we target the TACO v1.3 10s10t traces dataset
trace_dataset_pattern = "v1.3/10s10t.*of000026.*.lmdb"  # specific pattern for the v1.3 dataset
expected_part_count = 26  # given the 500-problem-chunk split used for the v1.3 dataset
selected_part_indices: int | list[int] = list(range(13))  # list of zero-based indices
target_datamodule = "shortcuts"
target_subsets = ["train", "valid"]
max_samples_per_subset = None  # None = no maximum
seed = 0
# ---------------------------------------------------------

In [ ]:
# prepare the target datamodule with its default config given the provided dataset info
dataset_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name=source_dataset_name,
        pattern=trace_dataset_pattern,
    )
)
if isinstance(selected_part_indices, int):
    selected_part_indices = [selected_part_indices]
assert isinstance(selected_part_indices, list) and len(selected_part_indices) > 0, "missing shard selection"
assert all(0 <= idx < len(dataset_paths) for idx in selected_part_indices), "invalid shard selection"
dataset_paths = [dataset_paths[idx] for idx in selected_part_indices]
print("will perform analysis on the following dataset shards:")
for path in dataset_paths:
    print(f"- {path}")
supported_datamodules = ["shortcuts"]
if target_datamodule not in supported_datamodules:
    raise ValueError(f"unsupported datamodule: {target_datamodule}; pick one of {supported_datamodules}")
elif target_datamodule == "shortcuts":
    default_dm_config = shortcuts_configs.get_datamodule_config(
        lmdb_paths=dataset_paths,
        split_file_path=pyine.data.utils.splits.get_dataset_split_file_path(source_dataset_name),
        seed=seed,
        as_pydantic=True,
    )
    datamodule = shortcuts.ShortcutBiasDataModule(default_dm_config, verbose=True)
else:
    raise NotImplementedError(f"missing impl for datamodule: {target_datamodule}")
print("performing datamodule preparation + setup...")
datamodule.prepare_data()
datamodule.setup()
for subset_name in target_subsets:
    parser = datamodule.get_parser(subset_name)
    print(f"- {subset_name} parser possesses {len(parser)} samples")
print("datamodule ready-to-go!")

In [ ]:
def collect_sample_rows(
    builder: sample_utils.SampleBuilder,
    subset_name: str,
    max_samples: int | None = None,
) -> list[dict[str, typing.Any]]:
    """Collect tabular rows describing generated samples.

    Args:
        builder: SampleBuilder to draw samples from.
        subset_name: Subset name associated with the builder.
        max_samples: Optional cap on the number of samples to collect.

    Returns:
        List of dictionaries containing summary information for each sampled entry.
    """
    sample_rows: list[dict[str, typing.Any]] = []
    sample_cap = len(builder) if max_samples is None else min(len(builder), max_samples)
    sample_idx_iter = tqdm.tqdm(range(sample_cap), desc="parsing samples", smoothing=0.1)
    for sample_idx in sample_idx_iter:
        sample = builder[sample_idx]
        tag_list = [tag for tag in sample.comma_separated_tags.split(",") if tag]
        sample_rows.append(
            {
                "subset": subset_name,
                "identifier": sample.identifier,
                "code_line_count": len(sample.code.splitlines()),
                "code_target_line_span": sample.last_line - sample.first_line,
                "description_word_count": len(sample.description.split()),
                "inputs_char_length": len(sample.inputs),
                "expected_output_char_length": len(sample.expected_output),
                "output_type": sample.output_type,
                "code_type": sample.code_type,
                "trace_step_count": sample.trace_step_count,
                "has_code_override": sample.has_code_override,
                "tags": tag_list,
            }
        )
    return sample_rows


builder_map: dict[str, sample_utils.SampleBuilder] = {}
sample_rows: list[dict[str, typing.Any]] = []
for subset_name in target_subsets:
    parser = datamodule.get_parser(subset_name)
    assert isinstance(parser, sample_utils.SampleBuilder), f"unexpected parser type: {type(parser)}"
    if not len(parser):
        print(f"Skipping {subset_name}: no samples found.")
        continue
    builder_map[subset_name] = parser
    print(f"collecting samples for {subset_name} subset...")
    rows = collect_sample_rows(
        builder=parser,
        subset_name=subset_name,
        max_samples=max_samples_per_subset,
    )
    sample_rows.extend(rows)
print(f"sample collection complete (total samples: {len(sample_rows)})")

In [ ]:
if sample_rows:
    samples_df = pd.DataFrame(sample_rows)
    ipy_display.display(samples_df.head())
else:
    samples_df = pd.DataFrame()
    print("no samples collected; ensure the dataset is available and contains traces")

In [ ]:
if builder_map:
    stats_rows = []
    expected_probs_map = {}
    for subset_name, builder in builder_map.items():
        stats = builder.get_stats()
        stats_rows.append({"subset": subset_name, **stats})
        expected_probs_map[subset_name] = builder.selection_config.input_type_prob_map
    stats_df = pd.DataFrame(stats_rows).set_index("subset").fillna(0)
    ipy_display.display(stats_df)

    # also create a df for ratio'd statistics
    prefixes_to_skip = ["sample_count", "code_summaries_count", "filtering"]
    prefixes_to_ratio = ["code_type_counts/", "code_overrides_count"]
    ratios_df_cols = [c for c in stats_df.columns if not any(c.startswith(s) for s in prefixes_to_skip)]
    ratios_df_target_cols = [c for c in ratios_df_cols if any(c.startswith(s) for s in prefixes_to_ratio)]
    ratios_df = pd.DataFrame(stats_df[ratios_df_cols], index=stats_df.index)
    ratios_df[ratios_df_target_cols] = ratios_df[ratios_df_target_cols].div(stats_df["sample_count"].squeeze(), axis=0)
    ratios_df = ratios_df.rename(columns=lambda c: c.replace("count", "ratio"))
    expected_prob_rows = [
        {f"code_type_ratios/{input_type}": prob for input_type, prob in expected_probs_map[subset_name].items()}
        for subset_name in target_subsets
    ]
    ratios_df = pd.concat(
        [
            ratios_df,
            pd.DataFrame(expected_prob_rows, index=[f"{s}_expected" for s in target_subsets]),
        ]
    )
    ratios_df = ratios_df.sort_index()
    ipy_display.display(ratios_df)
else:
    print("no builds created; statistics are unavailable")

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    metric_definitions = [
        ("code_line_count", "Code length (lines)", {"bins": 30}),
        ("code_target_line_span", "Target code line span", {"bins": 30}),
        ("description_word_count", "Description length (words)", {"bins": 30}),
        ("inputs_char_length", "Input args length (chars)", {"bins": 50, "log": True}),
        (
            "expected_output_char_length",
            "Expected output length (chars)",
            {"bins": 50, "log": True},
        ),
        ("trace_step_count", "Trace step count", {"bins": 50, "log": True}),
    ]
    for metric_column, metric_title, plot_kwargs in metric_definitions:
        fig, axes = plt.subplots(
            1,
            len(target_subsets),
            figsize=(6 * len(target_subsets), 4),
            sharey=True,
        )
        if len(target_subsets) == 1:
            axes = [axes]
        for axis, subset_name in zip(axes, target_subsets, strict=False):
            subset_df = samples_df[samples_df["subset"] == subset_name]
            if subset_df.empty:
                axis.text(0.5, 0.5, "No samples", ha="center", va="center")
                axis.set_title(f"[{subset_name}]")
                axis.set_xlabel(metric_title)
                axis.set_ylabel("Sample count")
                continue
            axis.hist(subset_df[metric_column], color="#2a9d8f", alpha=0.85, **plot_kwargs)
            axis.set_title(f"[{subset_name}]")
            axis.set_xlabel(metric_title)
            axis.set_ylabel("Sample count")
            axis.set_yscale("log")
        if len(target_subsets) > 1:
            fig.suptitle(metric_title)
        plt.tight_layout()
        plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    output_counts = samples_df.groupby(["subset", "output_type"]).size().unstack(fill_value=0).sort_index(axis=1)
    ax = output_counts.plot(kind="bar", figsize=(10, 6))
    plt.title("Output type distribution by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = output_counts.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    code_type_counts = samples_df.groupby(["subset", "code_type"]).size().unstack(fill_value=0).sort_index(axis=1)
    ax = code_type_counts.plot(kind="bar", figsize=(10, 6))
    plt.title("Code type distribution by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = code_type_counts.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    tag_counter = collections.Counter()
    for tag_list in samples_df["tags"]:
        tag_counter.update(tag_list)
    most_common_tags = tag_counter.most_common(20)
    if not most_common_tags:
        print("no tags identified in the sampled data")
    else:
        tag_labels, tag_values = zip(*most_common_tags, strict=False)
        total_samples = len(samples_df)
        proportions = [v / total_samples for v in tag_values]

        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.bar(tag_labels, tag_values, color="#264653", alpha=0.9)
        ax.set_ylabel("Occurrences")
        ax.set_title("Top 20 tags across sampled data")
        ax.set_xticks(range(len(tag_labels)))
        ax.set_xticklabels(tag_labels, rotation=45, ha="right")

        percent_labels = [f"{p * 100:.1f}%" for p in proportions]
        ax.bar_label(bars, labels=percent_labels, padding=3, fontsize=9, color="#1d3557")

        ymax = max(tag_values) if tag_values else 1
        ax.set_ylim(0, ymax * 1.15)
        ax.grid(axis="y", linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    override_summary = (
        samples_df.groupby(["subset", "has_code_override"])
        .size()
        .unstack(fill_value=0)
        .rename(
            columns={
                False: "no override",
                True: "with override",
            }
        )
    )
    ax = override_summary.plot(kind="bar", figsize=(8, 5))
    plt.title("Code override counts by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = override_summary.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()